# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

Data represents 17 campaigns

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [3]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [5]:
df['y'].value_counts(normalize=True)

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

***Business objective*** of this task is to find the best model that can explain the success of the bank marketing campaigns and the attributes that lead to the campaign's success

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [6]:
bank_info = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'y']
df_bank = df[bank_info]
df_bank.head()

,age,job,marital,education,default,housing,loan,y
0,56,housemaid,married,basic.4y,no,no,no,no
1,57,services,married,high.school,unknown,no,no,no
2,37,services,married,high.school,no,yes,no,no
3,40,admin.,married,basic.6y,no,no,no,no
4,56,services,married,high.school,no,no,yes,no


In [7]:
df_bank.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        41188 non-null  int64 
 1   job        41188 non-null  object
 2   marital    41188 non-null  object
 3   education  41188 non-null  object
 4   default    41188 non-null  object
 5   housing    41188 non-null  object
 6   loan       41188 non-null  object
 7   y          41188 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.5+ MB


In [8]:
for a in bank_info[1:]:
    print(f'{a}: *********')
    print(df_bank[a].value_counts(normalize=True))

job: *********
job
admin.           0.253035
blue-collar      0.224677
technician       0.163713
services         0.096363
management       0.070992
retired          0.041760
entrepreneur     0.035350
self-employed    0.034500
housemaid        0.025736
unemployed       0.024619
student          0.021244
unknown          0.008012
Name: proportion, dtype: float64
marital: *********
marital
married     0.605225
single      0.280859
divorced    0.111974
unknown     0.001942
Name: proportion, dtype: float64
education: *********
education
university.degree      0.295426
high.school            0.231014
basic.9y               0.146766
professional.course    0.127294
basic.4y               0.101389
basic.6y               0.055647
unknown                0.042027
illiterate             0.000437
Name: proportion, dtype: float64
default: *********
default
no         0.791201
unknown    0.208726
yes        0.000073
Name: proportion, dtype: float64
housing: *********
housing
yes        0.523842
no   

***Data Prepration 1*** 

There are unknown values for each of the attributes which do no provide any information. Also the proportion of these rows are comparatively quite small. Drop the rows which have unknown values

In [14]:
df_bank_clean = df_bank.loc[(df['age'] != 'unknown') & 
                            (df['job'] != 'unknown') &
                            (df['marital'] != 'unknown') &
                            (df['education'] != 'unknown') &
                            (df['default'] != 'unknown') &
                            (df['housing'] != 'unknown') &
                            (df['loan'] != 'unknown')]
df_bank_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30488 entries, 0 to 41187
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        30488 non-null  int64 
 1   job        30488 non-null  object
 2   marital    30488 non-null  object
 3   education  30488 non-null  object
 4   default    30488 non-null  object
 5   housing    30488 non-null  object
 6   loan       30488 non-null  object
 7   y          30488 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.1+ MB


In [10]:
for a in bank_info[1:]:
    print(f'{a}: *********')
    print(df_bank_clean[a].value_counts(normalize=True))

job: *********
job
admin.           0.286572
blue-collar      0.186139
technician       0.179513
services         0.093709
management       0.075800
retired          0.039885
self-employed    0.035817
entrepreneur     0.035719
unemployed       0.024206
housemaid        0.022632
student          0.020008
Name: proportion, dtype: float64
marital: *********
marital
married     0.573734
single      0.309728
divorced    0.116538
Name: proportion, dtype: float64
education: *********
education
university.degree      0.341511
high.school            0.252526
professional.course    0.141728
basic.9y               0.140252
basic.4y               0.078064
basic.6y               0.045559
illiterate             0.000361
Name: proportion, dtype: float64
default: *********
default
no     0.999902
yes    0.000098
Name: proportion, dtype: float64
housing: *********
housing
yes    0.541885
no     0.458115
Name: proportion, dtype: float64
loan: *********
loan
no     0.843611
yes    0.156389
Name: proporti

In [11]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer

In [15]:
education_order = [
    'illiterate',
    'basic.4y',
    'basic.6y',
    'basic.9y',
    'high.school',
    'professional.course',
    'university.degree'
]

transformer = make_column_transformer((OneHotEncoder(drop='if_binary', sparse_output=False), ['job', 'marital', 'default', 'housing', 'loan', 'y']),
                               (OrdinalEncoder(categories=[education_order]), ['education']),
                                remainder='passthrough')
t_data = transformer.fit_transform(df_bank_clean)
feature_names = [feature.split('__')[1] for feature in transformer.get_feature_names_out()]
df_bank_clean = pd.DataFrame(t_data, columns=feature_names)



In [16]:

df_bank_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30488 entries, 0 to 30487
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_admin.         30488 non-null  float64
 1   job_blue-collar    30488 non-null  float64
 2   job_entrepreneur   30488 non-null  float64
 3   job_housemaid      30488 non-null  float64
 4   job_management     30488 non-null  float64
 5   job_retired        30488 non-null  float64
 6   job_self-employed  30488 non-null  float64
 7   job_services       30488 non-null  float64
 8   job_student        30488 non-null  float64
 9   job_technician     30488 non-null  float64
 10  job_unemployed     30488 non-null  float64
 11  marital_divorced   30488 non-null  float64
 12  marital_married    30488 non-null  float64
 13  marital_single     30488 non-null  float64
 14  default_yes        30488 non-null  float64
 15  housing_yes        30488 non-null  float64
 16  loan_yes           304

In [24]:
df_bank_clean.rename(columns={'y_yes':'y'}, inplace=True)

In [25]:
df_bank_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30488 entries, 0 to 30487
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_admin.         30488 non-null  float64
 1   job_blue-collar    30488 non-null  float64
 2   job_entrepreneur   30488 non-null  float64
 3   job_housemaid      30488 non-null  float64
 4   job_management     30488 non-null  float64
 5   job_retired        30488 non-null  float64
 6   job_self-employed  30488 non-null  float64
 7   job_services       30488 non-null  float64
 8   job_student        30488 non-null  float64
 9   job_technician     30488 non-null  float64
 10  job_unemployed     30488 non-null  float64
 11  marital_divorced   30488 non-null  float64
 12  marital_married    30488 non-null  float64
 13  marital_single     30488 non-null  float64
 14  default_yes        30488 non-null  float64
 15  housing_yes        30488 non-null  float64
 16  loan_yes           30488 no

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [26]:
from sklearn.model_selection import train_test_split

In [38]:
X, y = df_bank_clean.drop('y', axis=1), df_bank_clean['y']
X.head()

,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,marital_divorced,marital_married,marital_single,default_yes,housing_yes,loan_yes,education,age
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,56.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,4.0,37.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,40.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,4.0,56.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,5.0,59.0


In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, stratify=y)
X_train = X_train.values
X_test = X_test.values
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((20426, 19), (20426,), (10062, 19), (10062,))

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

***Baseline***

Use the basic majority class proportion as baseline, i.e. if we randomly guess the class based on the proportion of the classes, we would get 87% success. So the trained classifier models should beat at least this baseline which is 87%

normalized value_counts of y column:

    no     0.873426
    yes    0.126574

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [70]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from time import time

In [76]:
lr_pipe = Pipeline([('scaler', StandardScaler()),
                    ('lr', LogisticRegression())])
lr_start_time = time()
lr_pipe.fit(X_train, y_train)
lr_end_time = time()
lr_train_time = lr_end_time - lr_start_time

### Problem 9: Score the Model

What is the accuracy of your model?

In [77]:
lr_train_accuracy = lr_pipe.score(X_train, y_train)
lr_test_accuracy = lr_pipe.score(X_test, y_test)
lr_train_time, lr_train_accuracy, lr_test_accuracy

(0.031225204467773438, 0.8734456085381377, 0.8733850129198967)

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [103]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [106]:
classifiers = [LogisticRegression(), KNeighborsClassifier(), DecisionTreeClassifier(), SVC()]
model = ['LogisticRegression', 'KNearestNeighbors', 'DecisionTree', 'SVM']
train_time = []
train_accuracy = []
test_accuracy = []

for classifier in classifiers:
    pipe = Pipeline([('scaler', StandardScaler()), 
                    ('classifier', classifier)])
    start_time = time()
    pipe.fit(X_train, y_train)
    end_time = time()
    train_time.append(end_time - start_time)
    train_accuracy.append(pipe.score(X_train, y_train))
    test_accuracy.append(pipe.score(X_test, y_test))

train_time, train_accuracy, test_accuracy

([0.027957916259765625,
  0.0047838687896728516,
  0.026370763778686523,
  2.8765811920166016],
 [0.8734456085381377,
  0.8771663566043278,
  0.9038969940272202,
  0.8734945657495349],
 [0.8733850129198967,
  0.8631484794275492,
  0.8512224209898629,
  0.8733850129198967])

In [110]:
model_df = pd.DataFrame({'Model': model,
                         'Train Time': train_time,
                         'Train Accuracy': train_accuracy,
                         'Test Accuracy': test_accuracy})
#model_df = model_df.set_index('Model')
model_df

,Model,Train Time,Train Accuracy,Test Accuracy
0,LogisticRegression,0.027958,0.873446,0.873385
1,KNearestNeighbors,0.004784,0.877166,0.863148
2,DecisionTree,0.026371,0.903897,0.851222
3,SVM,2.876581,0.873495,0.873385


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

##### Questions